# Overview
MetaCLIP is an improved way of training image-text models like CLIP. CLIP models learn by matching an image with the text that describes it, but large web datasets usually have noisy or inaccurate captions that hurt performance. MetaCLIP fixes this by using a better filtering and matching process to create image-text pairings that are more accurate. By improving the quality of the training data, the model learns stronger and more reliable connections, improving its overall accuracy even on very large datasets.

To test if the model is being processed using the GPU, run
```
import torch
print(torch.cuda.is_available())
```
If the output is True, then the model is going to be processed using the GPU. Processing using the CPU will work however, it will take 10x the amount of time compared to running it on the GPU.

## Packages
These are the necessary packages needed for the code to run.

#### warnings
It suppresses annoying warning messages.
#### collections
Calculates the frequency of an item in a datatype.
#### matplotlib
It is used to create visualizations of the data through plots, graphs, and charts.
#### sklearn.metrics
It is a toolbox for machine learning resources for training, evaluations, and preparing data. In this code, it was used for evaluating the performance through giving scores.
#### torch
It is used for optimizing the code and making it run on the GPU in order to have faster results. In this case, it uses computer vision. It provides a suite of tools that simplify the process of loading, processing, and building deep learning models for the images.
#### gc
It allows for direct access into Python's memory clean up.

Use ``` %%capture ``` to not see downloads

In [1]:
!pip install evaluate datasets accelerate
!pip install transformers torchvision
!pip install huggingface-hub hf_xet imbalanced-learn

import warnings
warnings.filterwarnings("ignore")

import gc
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import evaluate
from transformers import (
    TrainingArguments,
    Trainer,
    AutoImageProcessor,
    AutoModelForImageClassification
)
import torch
from torchvision.transforms import (
  Compose,
  Normalize,
  RandomRotation,
  RandomAdjustSharpness,
  Resize,
  ToTensor
)

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


## Database Information

### Data Collection:
The images in the dataset come from different models and types of cameras, which can affect the visual appearance of left vs. right. Some images are shown as one would see the retina anatomically (macula on the left, optic nerve on the right for the right eye). Others are shown as one would see through a microscope condensing lens (i.e. inverted, as one sees in a typical live eye exam).


### Resizing:
Images resized to 1024x1024

### Creators:
EyePACS

Emma Dugas, Jared, Jorge, and Will Cukierski
### Citation:
Emma Dugas, Jared, Jorge, and Will Cukierski. Diabetic Retinopathy Detection. https://kaggle.com/competitions/diabetic-retinopathy-detection, 2015. Kaggle.


### License

MIT License

Copyright (c) [year] [fullname]

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

---

### Notebooks
A notebook was used upload the dataset.
1. Go to huggingface.co and sign in
2. Click your profile picture in the top right
3. Click Settings
4. Click Access Tokens in the left sidebar
5. Click New Token
6. Give it any name (e.g. "kaggle")
7. Set role to Read
8. Click Create Token
9. Copy the token (it starts with `hf_...`)

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from datasets import load_dataset, Dataset, ClassLabel
ds_train = load_dataset("bumbledeep/eyepacs", split="train")

file_names = []
raw_labels = []
for i in ds_train:
  file_names.append(str(i["image"]))
  raw_labels.append(i["label_code"])

print(f"Loaded {len(file_names)} images")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/467M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/train-00003-of-00014.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

...

data/train-00011-of-00014.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00012-of-00014.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/train-00013-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/35108 [00:00<?, ? examples/s]

Loaded 35108 images


## Dataset Imbalance

The dataset is heavily imbalanced with majority of the images being of no diabetic retinopathy (labeled as id 0) at 73.3%. A table is created for the images and it's labels. Based on that `RandomOverSampler` checks which of those labels are unbalanced and duplicates the images that are underrepresented.

For example, severe diabetic retinopathy images are only 2.5% of the entire database. These images are duplicated so they can be relatively even to images that show no diabetic retinopathy.

In [4]:
df = pd.DataFrame({"img": file_names, "label": raw_labels})
print("Before oversampling:", Counter(df["label"]))

y = df[["label"]]
df = df.drop(["label"], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
df["label"] = y_resampled
del y, y_resampled
gc.collect()
print("After oversampling:", Counter(df["label"]))

Before oversampling: Counter({0: 25802, 2: 5288, 1: 2438, 3: 872, 4: 708})
After oversampling: Counter({0: 25802, 1: 25802, 2: 25802, 4: 25802, 3: 25802})


## Label Mappings

Here, the labels are provided with an id. A list is intially created and then a for loop is made to give each label an id and each id a label, This essentially creates two very similar lists where the keys and values are swapped. The object `ClassLabel` is build by HuggingFace in order formalize and allow HuggingFace features to be properly used.

### Useful print statement(s)
```
print("Mapping of IDs to Labels:", id2label)
print("Mapping of Labels to IDs:", label2id)
```



In [5]:
labels_list = [
  "no_diabetic_retinopathy",
  "mild_retinopathy",
  "moderate_retinopathy",
  "severe_retinopathy",
  "proliferative_retinopathy"
]

label2id, id2label = {}, {}
for i, label in enumerate(labels_list):
  label2id[label] = i
  id2label[i] = label

ClassLabels = ClassLabel(num_classes=5, names=labels_list)

# Building and Splitting the Dataset

The dataset is loaded and split for testing and training. The testing size is 30% and is shuffled in order to provide better results and reduce pattern guessing.

### Useful print statement(s)
```
print(f"Train: {len(train_data)}")
print(f"Test: {len(test_data)}")
```



In [6]:
ds = load_dataset("bumbledeep/eyepacs", split="train")  # single split only
ds = ds.cast_column("label_code", ClassLabels)
ds = ds.train_test_split(test_size=0.3, shuffle=True, stratify_by_column="label_code")

train_data = ds["train"]
test_data = ds["test"]

Casting the dataset:   0%|          | 0/35108 [00:00<?, ? examples/s]

# Processor and Transformations
`_train_transforms`: It randomly rotates the images and adjusts the sharpness. This is so the model can be trained on many variabilities that it can encounter.  

`_val_transforms`: Similar to the previous variable, it removes the variability in order to check if the model is correctly processing the images. THe randomness is left out in order to see how the model performs on unmodified images.

`train_transforms(items)`: Every image is converted to RGB coloring and then trained with variability. The image tensor is then stored as pixel values.

`val_transforms(items)`: The same thing happens as the previous function, however, the variability aspect is removed.

In [7]:
model_str = "facebook/metaclip-2-worldwide-s16"
processor = AutoImageProcessor.from_pretrained(model_str)

image_mean = processor.image_mean
image_std = processor.image_std
size = processor.size["height"]

_train_transforms = Compose([
  Resize((size, size)),
  RandomRotation(90),
  RandomAdjustSharpness(2),
  ToTensor(),
  Normalize(mean=image_mean, std=image_std)
])

_val_transforms = Compose([
  Resize((size, size)),
  ToTensor(),
  Normalize(mean=image_mean, std=image_std)
])

def train_transforms(items):
  items["pixel_values"] = [_train_transforms(image.convert("RGB")) for image in items["image"]]
  return items

def val_transforms(items):
  items["pixel_values"] = [_val_transforms(image.convert("RGB")) for image in items["image"]]
  return items

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

preprocessor_config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


# Collate Function
All of the pixel values and tensor values into one dictionary, so the images can be processed in batches.

In [8]:
def collate_fn(items):
    pixel_values = torch.stack([item["pixel_values"] for item in items])
    labels = torch.tensor([item["label_code"] for item in items])
    return {"pixel_values": pixel_values, "labels": labels}

# The Model
Here, the model is set up for configuration and specified only for image classification. MetaCLIP can do both text and image classification with newer updates, however for this situation, only image classification is needed.
### Useful print statement(s)
```
print(f"Trainable parameters: {model.num_parameters(only_trainable=True) / 1e6:.1f}M")
```

In [9]:
model = AutoModelForImageClassification.from_pretrained(
  model_str,
  num_labels=5,
  ignore_mismatched_sizes=True
)

model.config.id2label = id2label
model.config.label2id = label2id

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MetaClip2ForImageClassification LOAD REPORT from: facebook/metaclip-2-worldwide-s16
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPE

# Metrics

Each of the metrics for evaluation are determined when the prediction is inputted. It determines the accuracy of each of the predictions.

In [10]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  predictions = eval_pred.predictions.argmax(axis=1)
  references = eval_pred.label_ids
  acc = accuracy_metric.compute(predictions=predictions, references=references)["accuracy"]
  return {"accuracy": acc}

# Training Arguments

All the arguments for training the model are determined and stored for saving.

As previously mentioned with batches for the collate function, `per_device_train_batch_size=32` sets up train the training to be batched into 32 images.

The number of epochs for training is set to four. `num_train_epochs=4`

In [11]:
args = TrainingArguments(
    output_dir="metaclip-2-image-classification/",
    logging_dir="./logs",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.02,
    warmup_steps=50,
    remove_unused_columns=False,
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=4,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Trainer
The specifics on the model to be trained are determined and set.

In [12]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor
)

# Train and Evaluate
The final running is performed here.

Allowing resumation from checkpoint is useful in the case that the session times out or there is some later issue where the program stops running. In those situations, the training is saved and a checkpoint is setup so that when the code resumes, the training data is saved and nothing is lost.

In [14]:
trainer.train()
trainer.evaluate()

outputs = trainer.predict(test_data)
print(outputs.metrics)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.826390,0.712766,0.768347
2,0.673552,0.624525,0.792842
3,0.632090,0.595885,0.804139
4,0.581831,0.575461,0.811450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'test_loss': 0.5754612684249878, 'test_accuracy': 0.8114497294218171, 'test_runtime': 207.4599, 'test_samples_per_second': 50.771, 'test_steps_per_second': 6.348}
